# Community detection

This notebook introduces some fundamental community detection algorithms.
We explore the Girvan-Newman and Kernighan-Lin methods, compare their results, evaluate community quality, and examine modularity's limitations through the resolution limit phenomenon.

- The **Girvan-Newman** algorithm iteratively removes high-betweenness edges to reveal modular structure.  
- The **Kernighan-Lin** algorithm refines a bisection based on modularity optimization.  
- **Greedy modularity** and **label propagation** provide fast approximations for larger networks.  
- The **resolution limit** demonstrates modularity's inability to identify small communities in large networks.


## Setup

Setup and helper functions for modularity calculation, coverage, conductance, and simple plotting.

In [ ]:
# If needed, install dependencies (uncomment the next line)
# !pip install networkx matplotlib numpy

import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms.community.quality import modularity
from networkx.algorithms import cuts

def to_sets(communities):
    """Convert an iterable of iterables into a list of sets."""
    return [set(c) for c in communities]

def coverage_of(G, comms):
    """Compute the fraction of edges that fall within communities."""
    comms = to_sets(comms)
    intra_edges = 0
    for c in comms:
        intra_edges += G.subgraph(c).number_of_edges()
    return intra_edges / G.number_of_edges()

def modularity_of(G, comms):
    """Return the modularity of a given partition."""
    return modularity(G, to_sets(comms))

def avg_conductance(G, comms):
    """Compute the average conductance across all communities."""
    sets = to_sets(comms)
    vals = []
    for s in sets:
        if len(s) == 0 or len(s) == len(G):
            continue
        try:
            vals.append(cuts.conductance(G, s))
        except nx.NetworkXError:
            pass
    return sum(vals) / len(vals) if vals else float('nan')

def girvan_newman_modularity_curve(G):
    """Generate the Girvan–Newman sequence and compute modularity at each level."""
    comp_gen = nx.community.girvan_newman(G)
    levels, Qs, partitions = [], [], []
    for comp in comp_gen:
        comms = [tuple(sorted(c)) for c in comp]
        Q = modularity_of(G, comms)
        k = len(comms)
        levels.append(k)
        Qs.append(Q)
        partitions.append(comms)
        if k >= G.number_of_nodes():
            break
    return levels, Qs, partitions

def report_quality(label, G, comms):
    """Print modularity, coverage, and average conductance for a partition."""
    print(f"=== {label} ===")
    print("Modularity:", round(modularity_of(G, comms), 4))
    print("Coverage:", round(coverage_of(G, comms), 4))
    print("Average conductance:", round(avg_conductance(G, comms), 4))
    print()

def plot_graph(G, pos = None, title = "Graph", **kwargs):
    """Plot a network graph."""
    plt.figure()
    if pos is None:
        pos = nx.spring_layout(G, seed=7)
    nx.draw(G, pos, with_labels=True, node_size=300, font_size=8, **kwargs)
    plt.title(title)
    plt.axis('off')
    plt.show()

# Assign one color per community
def community_colors(G, communities):
    """Return a list of colors (one per node) based on community membership."""
    color_map = {}
    for i, comm in enumerate(communities):
        for node in comm:
            color_map[node] = i
    return [color_map[n] for n in G.nodes()]



## 1. Girvan–Newman Algorithm and Modularity Curve

The **Girvan–Newman** algorithm progressively removes edges with the highest betweenness centrality,
splitting the network into communities. The modularity value is recalculated at each step to evaluate the quality of the partition.


In [ ]:
# Karate club graph
G = nx.karate_club_graph()

# Compute modularity curve for Girvan–Newman
levels, Qs, partitions = girvan_newman_modularity_curve(G)

# Plot modularity versus number of communities
plt.figure()
plt.plot(levels, Qs, marker='o')
plt.xlabel("Number of communities (k)")
plt.ylabel("Modularity (Q)")
plt.title("Girvan–Newman: modularity vs. number of communities")
plt.show()

# Identify the partition with maximum modularity
best_idx = max(range(len(Qs)), key=lambda i: Qs[i])
best_partition_gn = partitions[best_idx]


## 2. Comparison: Girvan-Newman vs. Kernighan-Lin algorithm

The Kernighan-Lin algorithm optimizes modularity by iteratively swapping nodes between two groups to minimize cut size.
Here we compare its result with the best modularity partition from the Girvan-Newman method.


In [ ]:
# Girvan-Newman partition
partition_gn = best_partition_gn    # Best Q
# partition_gn = partitions[0]        # First

# Compute Kernighan–Lin bisection
S, T = nx.algorithms.community.kernighan_lin_bisection(G, seed=42)
partition_kl = [set(S), set(T)]

# Visual comparison using a fixed layout
# pos = nx.spring_layout(G, seed=42)
pos = nx.kamada_kawai_layout(G, weight=None)

# Compute color maps for each algorithm
colors_gn = community_colors(G, partition_gn)
colors_kl = community_colors(G, partition_kl)

# Plot with your helper
plot_graph(G, pos, title="Karate Club – Girvan–Newman (Best Q)", node_color=colors_gn, cmap=plt.cm.tab10)
plot_graph(G, pos, title="Karate Club – Kernighan–Lin Bisection", node_color=colors_kl, cmap=plt.cm.tab10)


## 3. Evaluating community quality

Community partitions can be evaluated using several complementary metrics:

- **Modularity (Q):** measures the density of edges inside communities relative to a null model.  
- **Coverage:** fraction of edges that fall within communities.  
- **Conductance:** quantifies how well communities are separated.

These metrics are computed for both the Girvan–Newman and Kernighan–Lin partitions.


In [ ]:
report_quality("Girvan–Newman (Best Q)", G, best_partition_gn)
report_quality("Kernighan–Lin", G, partition_kl)



## 4. Alternative algorithms: greedy modularity and label propagation

For larger graphs, more efficient heuristics are used to approximate good partitions.

- **Greedy modularity optimization** merges communities to maximize modularity incrementally.  
- **Label propagation** spreads labels among neighbors until communities stabilize.

Both are fast and typically yield results comparable to modularity-based methods.


In [ ]:
from networkx.algorithms.community import greedy_modularity_communities, label_propagation_communities

# --- Partitions ---
greedy_comms = list(greedy_modularity_communities(G))
lpa_comms    = list(label_propagation_communities(G))

# --- Colours ---
colors_greedy = community_colors(G, greedy_comms)
colors_lpa    = community_colors(G, lpa_comms)

# --- Visualization ---
plot_graph(G, pos, title="Karate Club – Greedy modularity",
           node_color=colors_greedy, cmap=plt.cm.tab10)

plot_graph(G, pos, title="Karate Club – Label propagation",
           node_color=colors_lpa, cmap=plt.cm.tab10)

# --- Quality metrics ---
report_quality("Greedy Modularity", G, greedy_comms)
report_quality("Label Propagation", G, lpa_comms)


## 5. Resolution limit: ring of cliques

The **resolution limit** describes modularity's tendency to merge small but well-defined communities into larger ones. This can be demonstrated using a *ring of cliques* network, where each clique is densely connected internally and sparsely connected to its neighbors.

In [ ]:
# 1) Build ring and compute partitions
n_cliques = 24        # try 24–40 for a clearer resolution-limit effect
clique_size = 4       # smaller cliques make merging more likely
G_rc = nx.ring_of_cliques(n_cliques, clique_size)

# Greedy modularity (modularity maximization)
rc_greedy = list(greedy_modularity_communities(G_rc))

# GN curve and best-Q
levels_rc, Qs_rc, parts_rc = girvan_newman_modularity_curve(G_rc)
best_idx_rc = max(range(len(Qs_rc)), key=lambda i: Qs_rc[i])
rc_gn_best = parts_rc[best_idx_rc]

# "Ground truth": partition with exactly k = n_cliques (each clique as a community)
def partition_by_k(levels, partitions, k):
    for kk, pp in zip(levels, partitions):
        if kk == k:
            return pp
    return None

rc_truth = partition_by_k(levels_rc, parts_rc, n_cliques)
assert rc_truth is not None, "Could not find k = n_cliques level in GN sequence."

# 2) Print quick summary
print(f"True number of cliques: {n_cliques}")
print("Detected communities:")
print("  Greedy modularity      :", len(rc_greedy))
print("  GN (best-Q)            :", len(rc_gn_best))
print("  GN (k = n_cliques, GT) :", len(rc_truth))

# 3) Visualize on the same layout
# pos_rc = nx.kamada_kawai_layout(G_rc, weight=None)
pos_rc = nx.forceatlas2_layout(G_rc)

plot_graph(G_rc, title="Ring – Ground Truth (k = n_cliques)",
           node_color=community_colors(G_rc, rc_truth), cmap=plt.cm.tab20)

plot_graph(G_rc, title="Ring – Greedy Modularity (modularity optimum)",
           node_color=community_colors(G_rc, rc_greedy), cmap=plt.cm.tab20)

plot_graph(G_rc, title="Ring – Girvan–Newman (Best Q)",
           node_color=community_colors(G_rc, rc_gn_best), cmap=plt.cm.tab20)

# 4) Quantify (optional)
report_quality("Ring – Ground Truth", G_rc, rc_truth)
report_quality("Ring – Greedy Modularity", G_rc, rc_greedy)
report_quality("Ring – GN (Best Q)", G_rc, rc_gn_best)
